# MS_B1 — Multi-Station HGB (Perfect Weather Forecast)

Trains one Optuna-tuned HGB model per horizon (h=1..24) for every station in `STATIONS_TO_RUN`.
Uses `weather_mode="perfect_forecast"` (MET_COLS shifted to t+h).

**Checkpoint:** skips a station if `outputs/{station}/results/B1_metrics.csv` already exists.
A kernel restart resumes from the last incomplete station.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import time
import joblib
import numpy as np
import pandas as pd
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import src.config as cfg
import src.data_loader as dl
import src.feature_engineering as fe

from src.config import ALL_STATIONS, HORIZONS, N_OPTUNA_TRIALS, CV_SPLITS, RANDOM_SEED, get_station_paths
from src.models.baseline_gbm import build_optuna_objective, _make_hgb
from src.evaluation import compute_metrics
from src.utils import ensure_dirs, set_seed

set_seed(RANDOM_SEED)

WEATHER_MODE = 'perfect_forecast'
# Override to run a subset, e.g. STATIONS_TO_RUN = ["MzWarChrosci"]
STATIONS_TO_RUN = ALL_STATIONS
# Set to a small number (e.g. 5) for a quick smoke-test
N_TRIALS = N_OPTUNA_TRIALS

print(f'Stations: {STATIONS_TO_RUN}')
print(f'Weather mode: {WEATHER_MODE} | Optuna trials: {N_TRIALS}')

In [ ]:
wall_start = time.time()
n = len(STATIONS_TO_RUN)

for i, station in enumerate(STATIONS_TO_RUN, 1):
    paths = get_station_paths(station)
    checkpoint = paths['results'] / 'B1_metrics.csv'

    if checkpoint.exists():
        print(f'[{i}/{n}] {station} — already done, skipping')
        continue

    print(f'\n[{i}/{n}] {station} — starting ...')
    t_station = time.time()

    ensure_dirs(paths['models'], paths['figures'], paths['results'])

    # Monkeypatch TARGET so all src functions use the current station
    cfg.TARGET = station
    dl.TARGET = station
    fe.TARGET = station

    df = dl.load_data()
    train_df, test_df = dl.train_test_split(df)

    hgb_models = {}

    # ── Train one HGB per horizon ─────────────────────────────────────
    for h in HORIZONS:
        X_tr, y_tr = fe.build_feature_matrix(train_df, horizon=h, weather_mode=WEATHER_MODE)

        study = optuna.create_study(
            direction='minimize',
            sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED)
        )
        study.optimize(
            build_optuna_objective(X_tr, y_tr, model_type='hgb'),
            n_trials=N_TRIALS,
            show_progress_bar=False,
        )

        model = _make_hgb(study.best_params)
        model.fit(X_tr.values, y_tr.values)

        model_path = paths['models'] / f'hgb_pfx_h{h}.pkl'
        joblib.dump(model, model_path)
        hgb_models[h] = model

    # ── Evaluate on test set ─────────────────────────────────────────
    rows = []
    for h in HORIZONS:
        X_te, y_te = fe.build_feature_matrix(test_df, horizon=h, weather_mode=WEATHER_MODE)
        preds = hgb_models[h].predict(X_te.values)
        m = compute_metrics(y_te.values, preds)
        rows.append({'Model': 'B1_HGB_pfx', 'Station': station, 'Horizon': h, **m})

    # Save metrics immediately so checkpoint is valid on next restart
    pd.DataFrame(rows).to_csv(checkpoint, index=False)

    elapsed_min = (time.time() - t_station) / 60
    total_elapsed_min = (time.time() - wall_start) / 60
    avg_per_station = total_elapsed_min / i
    remaining_min = avg_per_station * (n - i)
    print(f'[{i}/{n}] {station} — done | elapsed {elapsed_min:.1f}min | est. remaining {remaining_min:.1f}min')

# Restore original TARGET
cfg.TARGET = 'MzWarChrosci'
dl.TARGET = 'MzWarChrosci'
fe.TARGET = 'MzWarChrosci'

print(f'\nAll stations complete in {(time.time() - wall_start) / 60:.1f}min total.')